In [ ]:
rawgef_path='/data/input/Files/yangdong/M.truncatula/SAW/WT202604020036551/result/Y00710F6/outs/feature_expression/Y00710F6.raw.gef'
mask_tif='/data/work/0511/tissue_seg/wt1.tif'

In [17]:
import stereo as st
import cv2
import numpy as np

# 1. 加载原始 raw.gef 数据
# 这里的 bin_size 根据你的需求设置（如 1, 10, 50 等）
data = st.io.read_gef(file_path=rawgef_path, bin_size=1); print(data)

[2026-05-11 22:57:06][Stereo][369][MainThread][139747671279424][reader][1339][INFO]: read_gef begin ...


2026-05-11 22:57:06 [INFO   ] input file:/data/input/Files/yangdong/M.truncatula/SAW/WT202604020036551/result/Y00710F6/outs/feature_expression/Y00710F6.raw.gef specify block size:1


[2026-05-11 22:57:34][Stereo][369][MainThread][139747671279424][reader][1488][INFO]: the matrix has 20561036 cells, and 138690 genes.
[2026-05-11 22:57:34][Stereo][369][MainThread][139747671279424][reader][1489][INFO]: read_gef end.


StereoExpData object with n_cells X n_genes = 20561036 X 138690
bin_type: bins
bin_size: 1
offset_x = 0
offset_y = 0
cells: ['cell_name']
genes: ['gene_name', 'real_gene_name']
cells_matrix = ['spatial']
Layers with keys: 
tl.result: []


In [18]:
# def filter_mask(data, mask_img):
#     import numpy as np
#     ## filter cells

#     # 3. 提取组织区域的坐标
#     # 假设 mask 中 255 代表组织，0 代表环境
#     # 注意：图像坐标 (row, col) 与 GEF 坐标 (x, y) 的对应关系可能需要根据配准情况微调
#     # Stereopy 加载 GEF 后，data.position 存储了坐标信息
#     x_coords = data.position[:, 0].astype(int)
#     y_coords = data.position[:, 1].astype(int)

#     # 4. 根据 Mask 进行布尔过滤
#     # 检查每个基因点在 Mask 对应像素点的值是否大于 0
#     is_in_tissue = mask_img[y_coords, x_coords] > 0

#     # 将布尔数组 [True, False, True...] 转换为索引 [0, 2...]
#     selected_indices = np.where(is_in_tissue)[0]

#     # 使用正确的参数名 cell_index
#     data1 = data.sub_by_index(cell_index=selected_indices)

#     ## filter genes

#     # 1. 计算每个基因在所有剩余位点中的总表达量
#     # data.exp_matrix 通常是一个稀疏矩阵 (scipy.sparse)
#     # 我们按列（axis=0）求和得到每个基因的总 count
#     gene_counts = np.array(data1.exp_matrix.sum(axis=0)).flatten()

#     # 2. 找到表达量 > 0 的基因索引
#     nonzero_gene_indices = np.where(gene_counts > 0)[0]

#     # 3. 再次使用你熟悉的 sub_by_index 方法
#     # 这一次我们传入的是 gene_index
#     data1.sub_by_index(gene_index=nonzero_gene_indices)

#     # 4. 验证结果
#     print(f"过滤前基因数: {len(gene_counts)}")
#     print(f"过滤后基因数: {len(data1.gene_names)}")
#     print(data1)
#     return data1

In [19]:
# 2. 加载你在 Photoshop 中处理好的 tif 文件
mask_img = cv2.imread(mask_tif, cv2.IMREAD_GRAYSCALE)

In [20]:
data

StereoExpData object with n_cells X n_genes = 20561036 X 138690
bin_type: bins
bin_size: 1
offset_x = 0
offset_y = 0
cells: ['cell_name']
genes: ['gene_name', 'real_gene_name']
cells_matrix = ['spatial']
Layers with keys: 
tl.result: []

### filter cell

In [21]:
import numpy as np

# 3. 提取组织区域的坐标
# 假设 mask 中 255 代表组织，0 代表环境
# 注意：图像坐标 (row, col) 与 GEF 坐标 (x, y) 的对应关系可能需要根据配准情况微调
# Stereopy 加载 GEF 后，data.position 存储了坐标信息
x_coords = data.position[:, 0].astype(int)
y_coords = data.position[:, 1].astype(int)

# 4. 根据 Mask 进行布尔过滤
# 检查每个基因点在 Mask 对应像素点的值是否大于 0
is_in_tissue = mask_img[y_coords, x_coords] > 0

# 将布尔数组 [True, False, True...] 转换为索引 [0, 2...]
selected_indices = np.where(is_in_tissue)[0]

# 使用正确的参数名 cell_index
data.sub_by_index(cell_index=selected_indices)

StereoExpData object with n_cells X n_genes = 41659 X 138690
bin_type: bins
bin_size: 1
offset_x = 0
offset_y = 0
cells: ['cell_name']
genes: ['gene_name', 'real_gene_name']
cells_matrix = ['spatial']
Layers with keys: 
tl.result: []

### filter genes

In [22]:
import numpy as np

# 1. 计算每个基因在所有剩余位点中的总表达量
# data.exp_matrix 通常是一个稀疏矩阵 (scipy.sparse)
# 我们按列（axis=0）求和得到每个基因的总 count
gene_counts = np.array(data.exp_matrix.sum(axis=0)).flatten()

# 2. 找到表达量 > 0 的基因索引
nonzero_gene_indices = np.where(gene_counts > 0)[0]

# 3. 再次使用你熟悉的 sub_by_index 方法
# 这一次我们传入的是 gene_index
data.sub_by_index(gene_index=nonzero_gene_indices)

# 4. 验证结果
print(f"过滤前基因数: {len(gene_counts)}")
print(f"过滤后基因数: {len(data.gene_names)}")
print(data)

过滤前基因数: 138690
过滤后基因数: 14581
StereoExpData object with n_cells X n_genes = 41659 X 14581
bin_type: bins
bin_size: 1
offset_x = 0
offset_y = 0
cells: ['cell_name']
genes: ['gene_name', 'real_gene_name']
cells_matrix = ['spatial']
Layers with keys: 
tl.result: []


### save

In [23]:
import os 
prefix = os.path.basename(mask_tif)

In [24]:
# save the data, only the result after filtering
st.io.write_mid_gef(
        data=data,
        output=prefix+'_flt.gef'
        )

[2026-05-11 23:07:43][Stereo][369][MainThread][139747671279424][writer][416][INFO]: The output standard gef file only contains one expression matrix with mid count.Please make sure the expression matrix of StereoExpData object is mid count without normaliztion.
100%|██████████| 14581/14581 [00:41<00:00, 353.19it/s]
